# Curation of shebanq

By means of this notebook you can curate the personal data of users of the shebanq website.

Curation consists of the following steps

1. Obtain a database dump of the `shebanq_web` and `shebanq_note` databases on web server of the
   shebanq.ancient-data.org website
2. Create a TF set file containing the results of each query of the user, whether published, shared or private query as a separate set
3. Create a html page for the metadata of the user
4. Create html pages for his/her queries
5. Create pages for the query results
7. Create an index page for the query pages

# Base directory

We assume your base directory for the github clones of the ETCBC repos is as follows.

`BASEDIR` is the directory under which you have your github organizations, such as ETCBC.
And under organizations you have your repos.
So if repo `ETCBC/shebanq-local` resides under `/your/specific/directory`, you should set

```
BASEDIR = "/your/specific/directory"
```

In [1]:
BASEDIR = "~/github"

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from helpers import SQL, Check, Mapper

# Latest databases with user contributed data

We work with the same data backup that is the input for the [Curate](Curate.ipynb) notebook.

# Select the user data

From now on we work on the local computer, in the top-level directory of the clone
of the `shebanq-local` repo.

## Preparation

Make sure Docker (Desktop) is running.

Start the local shebanq in production mode:

```
./shebanq.sh up
```

In another shell, still in directory `~/github/ETCBC/shebanq-local` do

```
./shebanq.sh sh
```

Now you are in a shell of the local shebanq, from where you can restore the backup we just
fetched from the production shebanq:

```
cd src/scripts
./restore.sh ALL
```

Now we export the data as sql definition files plus tsv data files, by means of mysqldump with the `--tab` option.

Continue in the same shell and do

```
./export.sh
exit
```

Now there are two directories added to the `app/backup` folder, namely `shebanq_web` and `shebanq_note`, and they
contain the definition and data files for each table in that database.

Now you can explore the database by

```
./shebanq.sh browse admin
```

When asked for a password, it is `wajehior`.

### Map slots of all non-2021 versions to slots of 2021

As a next step in the preparation we load the mapping between BHSA versions as generated
by the [Curate](Curate.ipynb) notebook.

If you want to compute these mappings anew, pass `force=True`.

In [4]:
M = Mapper(BASEDIR)

In [5]:
M.loadMappings(force=False)

Mappings read from ~/github/ETCBC/shebanq-local/content/mappings.gz


## The selection process itself

Now we are in a position to select the data of the queries for a specific user from the exported files.

We start by reading the tables into rows of fields in Python

In [6]:
S = SQL(BASEDIR, zapTables={"web2py_session_shebanq", "auth_event", "auth_cas"})
S.stats()

Database shebanq_note:
	Table note                     :   222527 rows
Database shebanq_web:
	Table auth_group               :     1714 rows
	Table auth_membership          :     1709 rows
	Table auth_permission          :        0 rows
	Table auth_user                :     1710 rows
	Table monads                   :  6305406 rows
	Table organization             :      544 rows
	Table project                  :      801 rows
	Table query                    :     7141 rows
	Table query_exe                :     8253 rows
	Table uploaders                :        1 rows


### Queries en notes

We select all queries of a given user and store it in directory `content/userdata/user name`:

In [13]:
user = "Tony Jurg"
S.selectUserQueries(user)

Result directory is ~/github/ETCBC/shebanq-local/docsPrivate/user/Tony_Jurg


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,39,10938.21,100
chapter,929,459.19,100
lex,9230,46.22,100
verse,23213,18.38,100
half_verse,45179,9.44,100
sentence,63717,6.70,100
sentence_atom,64514,6.61,100
clause,88131,4.84,100
clause_atom,90704,4.70,100
phrase,253203,1.68,100


# Turn the user query results into a TF set

We save the query results of all these queries as a TF set in the user directory.

In [14]:
S.writeQResultsTF(M.mappingsFrom)

In [12]:
S.writeResultsHtml("12", "4")

In [30]:
S.genQueryPages()

Cleaning previous results ... 
Gathering projects ... 
Gathering organizations ... 
Gathering users ... 
Gathering queries ... 
Gathering query executions ... 
Generating pages ... 
Generated 1157 pages for 1130 queries
